In [2]:
import os  
print(os.getcwd())  # Prints the current working directory  

/Users/khanjanin/Desktop/Testing_the_codes


In [3]:
# import the Neo4j driver
from neo4j import GraphDatabase
driver = GraphDatabase.driver(uri = "bolt://34.217.248.7:5080", auth=None) # even the port is different on the website (7678), we should still use the old port 5080 

# print count of nodes and relationships using Neo4j driver
with driver.session() as session:
    node_count = session.run("MATCH (n) RETURN count(n) AS node_count").single().get("node_count")
    relationship_count = session.run("MATCH ()-[r]->() RETURN count(r) AS relationship_count").single().get("relationship_count")
    print(f"Node count: {node_count}")
    print(f"Relationship count: {relationship_count}")

Node count: 234367
Relationship count: 1693332


In [7]:
# create drug list
import pandas as pd 
data = pd.read_csv("/Users/khanjanin/Desktop/Testing_the_codes/output/QC_stats_100drugs.csv")  
drug_ids = data['Drug ID'].tolist()
#unique_drug_ids = list(set(drug_ids)) 
print(drug_ids)  # Print the drug IDs to verify
print(len(drug_ids))  # Print the total number of drug IDs to verify

[7842, 4255, 8196, 13241, 10310, 13719, 6564, 14129, 11838, 3718, 5592, 7516, 3765, 13657, 6217, 13091, 3662, 3719, 16286, 3497, 5624, 15480, 13067, 14201, 15164, 12305, 15401, 15942, 10511, 11951, 12035, 11668, 7334, 14854, 12726, 6562, 3491, 14812, 13069, 7547, 13413, 6592, 5553, 5810, 13532, 12538, 5023, 4677, 15895, 3794, 9826, 10208, 8417, 14202, 10546, 5363, 10686, 7048, 15497, 15379, 5815, 7551, 8105, 9764, 12041, 5377, 13435, 8435, 15026, 15262, 6258, 16026, 6077, 6398, 6725, 13044, 6485, 13064, 15020, 12975, 6580, 10553, 13782, 9870, 6484, 4579, 9358, 8637, 5490, 12608, 9851, 5558, 11469, 10396, 6435, 13570, 11532, 8660, 12718, 12549]
100


In [ ]:
#this query is to select drugs that are FDA approved and are associated with AD genes and brain regions.
#first we create a 3-hop drug to disease relationship.
#then we use the 3-hop relationship above with gene-brain regions relationship and drug-drugclass relationship

from neo4j import GraphDatabase
import pandas as pd  
  
# Connect to Neo4j database  
driver = GraphDatabase.driver(uri="bolt://34.217.248.7:5080", auth=None)  
  
# Define the list of selected drug IDs  
selected_drugs = [7842, 4255, 8196, 13241, 10310, 13719, 6564, 14129, 11838, 3718, 5592, 7516, 3765, 13657, 6217, 13091, 3662, 3719, 16286, 3497, 5624, 15480, 13067, 14201, 15164, 12305, 15401, 15942, 10511, 11951, 12035, 11668, 7334, 14854, 12726, 6562, 3491, 14812, 13069, 7547, 13413, 6592, 5553, 5810, 13532, 12538, 5023, 4677, 15895, 3794, 9826, 10208, 8417, 14202, 10546, 5363, 10686, 7048, 15497, 15379, 5815, 7551, 8105, 9764, 12041, 5377, 13435, 8435, 15026, 15262, 6258, 16026, 6077, 6398, 6725, 13044, 6485, 13064, 15020, 12975, 6580, 10553, 13782, 9870, 6484, 4579, 9358, 8637, 5490, 12608, 9851, 5558, 11469, 10396, 6435, 13570, 11532, 8660, 12718, 12549]
query = """  
UNWIND $drugIds AS drugId  
MATCH (d:Drug)-[]-(g:Gene)-[]-(ad:Disease {commonName: "Alzheimer's Disease"})  
OPTIONAL MATCH (g)-[]-(br:BrainRegion)
OPTIONAL MATCH (g)-[]-(t:TranscriptionFactor) 
OPTIONAL MATCH (d)-[]-(dc:DrugClass) 
WHERE id(d) = drugId  
RETURN   
    id(d) AS DrugId,   
    d.commonName AS DrugName,  
    COUNT(DISTINCT g) AS UniqueGenesConnectedToAD,  
    COUNT(DISTINCT CASE WHEN br IS NOT NULL THEN g END) AS GenesConnectedToADWithBrainRegions,  
    COUNT(DISTINCT CASE WHEN t IS NOT NULL THEN g END) AS GenesConnectedToADWithTranscriptionFactor,
    COUNT(DISTINCT dc) AS LinkedDrugClasses,
    COLLECT(DISTINCT CASE WHEN br IS NOT NULL THEN g.geneSymbol END) AS GeneSymbolsForBrainRegions,
    COLLECT(DISTINCT CASE WHEN g IS NOT NULL THEN g.geneSymbol END) AS AllADGeneSymbols,
    COLLECT(DISTINCT CASE WHEN br IS NOT NULL THEN br.commonName END) AS brainRegionsConnectedToADgenes
"""  
  
# Execute the query and process the results  
with driver.session() as session:  
    result = session.run(query, drugIds=selected_drugs)  
  
    # Create a list of dictionaries to store the results  
    data = []  
    for record in result:  
        data.append({  
            "DrugId": record["DrugId"],  
            "DrugName": record["DrugName"],  
            "UniqueGenesConnectedToAD": record["UniqueGenesConnectedToAD"],  
            "GenesConnectedToADWithBrainRegions": record["GenesConnectedToADWithBrainRegions"], 
            "GenesConnectedToADWithTranscriptionFactor": record["GenesConnectedToADWithTranscriptionFactor"],
            "LinkedDrugClasses": record["LinkedDrugClasses"],
            "GeneSymbolsForBrainRegions": record["GeneSymbolsForBrainRegions"],
            "AllADGeneSymbols": record["AllADGeneSymbols"],
            "brainRegionsConnectedToADgenes": record["brainRegionsConnectedToADgenes"]
        })  
  
# Close the driver  
driver.close()  
  
# Convert the data to a pandas DataFrame  
df = pd.DataFrame(data)  
  
# Ensure the order of the drugs in the DataFrame matches the `selected_drugs` list  
df["DrugId"] = pd.Categorical(df["DrugId"], categories=selected_drugs, ordered=True)  
df = df.sort_values("DrugId")  
  
# Print the DataFrame  
print(df)  
  
# Save the DataFrame to a CSV file  
output_file = "/Users/khanjanin/Desktop/Testing_the_codes/Drugs_associated_with_AD-gene_brain-region.csv"  
df.to_csv(output_file, index=False)  
  
print(f"Results saved to {output_file}") 

    DrugId     DrugName  UniqueGenesConnectedToAD  \
429   7842  Galantamine                         4   
260   8196  Haloperidol                         2   
544  13241   Quetiapine                         2   
795  10310    Memantine                         2   
362   6564    Donepezil                         6   
..     ...          ...                       ...   
850    NaN    Proguanil                         1   
851    NaN   Duloxetine                         2   
853    NaN  Oliceridine                         1   
854    NaN   Ranolazine                         2   
855    NaN     Naloxone                         2   

     GenesConnectedToADWithBrainRegions  \
429                                   3   
260                                   2   
544                                   2   
795                                   1   
362                                   6   
..                                  ...   
850                                   1   
851                